# Oil Spill - E2 weight sensitivity
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadDevX/ML-Research-Dataset-Imbalance/blob/main/oilspill/e2_weight_sensitivity/oilspill_e2_weight_sensitivity.ipynb)

Answers reviewer R3 (one weighting configuration only) and tests the over-correction mechanism.

* Classifiers: LR, RF, XGB.
* Resampling ratio: [0.0, 0.5, 1.0] (0 = none; otherwise SMOTETomek to that minority:majority ratio).
* Positive-class weight = IR ** power, power in [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
  (0 = none, 0.5 = sqrt-inverse, 1 = inverse frequency), plus weights recomputed after resampling.
* Each setting with and without threshold tuning (C).
* Hyperparameters are reused from `c0_none` (no resampling) and `c1_A_smotetomek` (with resampling).
  **Run those two notebooks first**; missing folds are tuned here, which is slower.
* Each row records `rho_eff`, the positive:negative loss mass the classifier sees.

**How to run on Colab**
1. Click the badge, then *Runtime > Run all*. Allow Google Drive access when asked.
2. Results are appended to Google Drive after every classifier/fold:
   `MyDrive/ML-Research-Dataset-Imbalance/oilspill/e2_weight_sensitivity/results/oilspill__e2_weight_sensitivity.csv`.
3. If the runtime disconnects, just *Run all* again: finished folds are skipped.
4. When it is complete, copy the CSV into the same path in the repository and commit it.

In [ ]:
# ---- Setup: identical in every notebook. Do not edit. ----
import os, sys, subprocess
REPO_URL = "https://github.com/MuhammadDevX/ML-Research-Dataset-Imbalance.git"
BRANCH = "main"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = "/content/ML-Research-Dataset-Imbalance"
    if not os.path.isdir(REPO_ROOT):
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_ROOT], check=True)
    else:
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{REPO_ROOT}/requirements.txt"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    # Results go to Google Drive so they survive a disconnect.
    os.environ.setdefault("RESULTS_ROOT", "/content/drive/MyDrive/ML-Research-Dataset-Imbalance")
else:
    REPO_ROOT = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(REPO_ROOT, "common")):
        parent = os.path.dirname(REPO_ROOT)
        if parent == REPO_ROOT:
            raise RuntimeError("Open this notebook from inside the repository.")
        REPO_ROOT = parent

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from common.env import check_environment
check_environment()


In [ ]:
DATASET = "oilspill"
# Quick test run (2 folds, fewer classifiers and candidates): change the next line
# to  SMOKE = True . Its results go to a separate *__SMOKE.csv file and must never
# be used in the paper. Leave it as it is for the real run.
SMOKE = os.environ.get("SMOKE", "0") == "1"

In [ ]:
from common.cv import run_e2
results = run_e2(DATASET, smoke=SMOKE)

## Macro-F1 by resampling ratio, weight setting and threshold tuning

In [ ]:
results.pivot_table(index=["sampling_ratio", "weight_mode", "weight_power"],
                    columns=["classifier", "C"], values="macro_f1").round(4)

In [ ]:
results.groupby(["sampling_ratio", "weight_mode", "weight_power"])[["rho_eff", "precision_1", "recall_1", "ppr_ratio"]].mean().round(3)